In [ ]:
# AUTHENTICATE & LOAD SHEETS
from google.colab import auth
auth.authenticate_user()

import time
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

spreadsheet = gc.open("Matching Algorithm")
company_sheet = spreadsheet.worksheet("Company info")
matches_sheet = spreadsheet.worksheet("Matches")

data = company_sheet.get_all_records()

# CLEAN & TYPE-CAST DATA
required_fields = ['Lowest Fixed', 'Lowest Floating', 'Loan size ', 'Loan term']
clean_data = []
for c in data:
    if all(str(c.get(f, '')).strip() != '' for f in required_fields):
        c['Company name']    = str(c['Company name'])
        c['Lowest Fixed']    = float(c['Lowest Fixed'])
        c['Lowest Floating'] = float(c['Lowest Floating'])
        c['Loan size ']      = float(c['Loan size '])
        c['Loan term']       = float(c['Loan term'])
        clean_data.append(c)
data = clean_data

# LOAN SAVINGS FUNCTION
def total_loan_savings(principal, old_rate, new_rate, years):
    monthly_old = old_rate / 100 / 12
    monthly_new = new_rate / 100 / 12
    n = years * 12
    old_payment = principal * (monthly_old * (1 + monthly_old) ** n) / ((1 + monthly_old) ** n - 1)
    new_payment = principal * (monthly_new * (1 + monthly_new) ** n) / ((1 + monthly_new) ** n - 1)
    return (old_payment - new_payment) * n

# SWAP SAVINGS SPLIT
SAVINGS_SPLIT = 0.5

# PAIR SAVINGS
def pair_savings(anchor, member, anchor_is_fixed, notional, term, split=SAVINGS_SPLIT):
    if anchor_is_fixed:
        pref        = anchor['Lowest Fixed']    + member['Lowest Floating']
        non_pref    = anchor['Lowest Floating'] + member['Lowest Fixed']
        anchor_base = anchor['Lowest Fixed']
        member_base = member['Lowest Floating']
    else:
        pref        = member['Lowest Fixed']    + anchor['Lowest Floating']
        non_pref    = member['Lowest Floating'] + anchor['Lowest Fixed']
        anchor_base = anchor['Lowest Floating']
        member_base = member['Lowest Fixed']
    benefit     = pref - non_pref
    anchor_save = total_loan_savings(notional, anchor_base, anchor_base - benefit * split,       term)
    member_save = total_loan_savings(notional, member_base, member_base - benefit * (1 - split), term)
    return benefit, anchor_save + member_save

# MEMBER SAVINGS RANK (clip-order key)
def member_savings_rate(member, anchor_is_fixed):
    spread = member['Lowest Fixed'] - member['Lowest Floating']
    return -spread if anchor_is_fixed else spread

# SHEET WRITE HELPER (429 backoff)
def sheet_write(func, *args, max_retries=6, **kwargs):
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except gspread.exceptions.APIError as e:
            if '429' in str(e) and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"Rate limit hit, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

# CONSTANTS
SIZE_LEEWAY   = 0.1
MIN_RATE_DROP = 0.25   # deal gate: each side's interest rate must drop at least this many pp

# REVENUE SPLIT
OUR_CUT      = 0.20
BANK_CUT     = 0.075
BROKER_CUT   = 0.125
BUSINESS_CUT = 0.45
RESERVES_CUT = 0.15

# PER-SIDE RATE DROP (pp) — each side's interest-rate drop from the swap.
# = (avg fixed-side spread − avg float-side spread) × SAVINGS_SPLIT, where
# spread = Lowest Fixed − Lowest Floating and all members on a side contribute equally.
def party_rate_drop(fixed_members, float_members):
    fx = sum(c['Lowest Fixed'] - c['Lowest Floating'] for c in fixed_members) / len(fixed_members)
    fl = sum(c['Lowest Fixed'] - c['Lowest Floating'] for c in float_members) / len(float_members)
    return (fx - fl) * SAVINGS_SPLIT

# SAVINGS COLUMN — the per-side rate drop in percentage points (e.g. 5.00% → 4.75% shows as 0.25).
# This is both the value written to the sheet and the deal-gate metric (must be >= MIN_RATE_DROP).
def displayed_savings(drop):
    return round(drop, 4)

# FIXED POOL RANKING
fixed_pool = sorted(
    [dict(c) for c in data if c['Preferred Loan Type'] == 'Fixed'],
    key=lambda c: (c['Lowest Fixed'] - c['Lowest Floating']) * c['Loan size '] * c['Loan term'],
    reverse=True
)

# FLOATING POOL RANKING
floating_candidates = [dict(c) for c in data if c['Preferred Loan Type'] == 'Floating']
min_raw_diff = min(c['Lowest Floating'] - c['Lowest Fixed'] for c in floating_candidates)
for c in floating_candidates:
    c['FloatScore'] = (c['Lowest Floating'] - c['Lowest Fixed'] - min_raw_diff + 1) * c['Loan size '] * c['Loan term']
floating_pool = sorted(floating_candidates, key=lambda c: c['FloatScore'], reverse=True)

# BUNDLE FORMATION
def try_form_bundle(anchor_size, candidates):
    lower  = anchor_size * (1 - SIZE_LEEWAY)
    upper  = anchor_size * (1 + SIZE_LEEWAY)
    bundle = []
    bundle_size = 0.0
    for loan in candidates:
        ls = loan['Loan size ']
        if ls > upper:
            continue
        bundle.append(loan)
        bundle_size += ls
        while bundle_size > upper and len(bundle) > 1:
            bundle_size -= bundle[0]['Loan size ']
            bundle.pop(0)
        if bundle_size > upper:
            bundle_size -= bundle[0]['Loan size ']
            bundle.pop(0)
            continue
        if bundle_size >= anchor_size:
            return bundle
    return bundle if (bundle and bundle_size >= lower) else None

# BUNDLE SAVINGS ($)
def bundle_savings(anchor, bundle, anchor_is_fixed):
    anchor_size  = anchor['Loan size ']
    remaining    = float(anchor_size)
    total_dollar = 0.0
    ordered = sorted(bundle, key=lambda l: member_savings_rate(l, anchor_is_fixed), reverse=True)
    for loan in ordered:
        if remaining <= 0:
            break
        notional  = min(loan['Loan size '], remaining)
        remaining -= notional
        term = min(anchor['Loan term'], loan['Loan term'])
        _, dollars = pair_savings(anchor, loan, anchor_is_fixed, notional, term)
        total_dollar += dollars
    return total_dollar

# ROW NAME PARSER
def match_names(m, col):
    return [n.strip() for n in str(m[col]).split(', ') if n.strip()]

# POST-MATCHING REBALANCING
def rebalance_connections(init_matches):
    if len(init_matches) < 2:
        return init_matches

    fx_rank = {c['Company name']: i for i, c in enumerate(fixed_pool)}
    fl_rank = {c['Company name']: i for i, c in enumerate(floating_pool)}
    fx_data = {c['Company name']: c for c in fixed_pool}
    fl_data = {c['Company name']: c for c in floating_pool}

    work = [list(m) for m in init_matches]

    directions = [
        (1, 0, fl_rank, fx_rank, fl_data, fixed_pool,    True),
        (0, 1, fx_rank, fl_rank, fx_data, floating_pool, False),
    ]

    improved = True
    extra_pass = True

    while improved or extra_pass:
        if not improved and extra_pass:
            extra_pass = False
        improved = False

        # ── PART 1: COVER UNMATCHED SPARSE COMPANIES ──
        for dom_col, sp_col, dom_rnk, sp_rnk, dom_dat, sp_pool_ord, dom_is_float in directions:

            sp_matched = {n for m in work for n in match_names(m, sp_col)}
            unmatched  = [c for c in sp_pool_ord if c['Company name'] not in sp_matched]
            if not unmatched:
                continue

            # Per-match cache (parsed dom names + min sparse-rank); rebuilt only when `work` changes
            def build_part1_cache():
                dom_names = [match_names(m, dom_col) for m in work]
                m_min_sp  = [min(sp_rnk.get(n, 999) for n in match_names(m, sp_col)) for m in work]
                return dom_names, m_min_sp
            dom_names_cache, m_min_sp_cache = build_part1_cache()
            cache_dirty = False

            for U in unmatched:
                if cache_dirty:
                    dom_names_cache, m_min_sp_cache = build_part1_cache()
                    cache_dirty = False

                U_name = U['Company name']
                U_rank = sp_rnk[U_name]
                U_size = U['Loan size ']
                lower  = U_size * (1 - SIZE_LEEWAY)
                upper  = U_size * (1 + SIZE_LEEWAY)

                eligible = sorted(
                    [
                        (dom_rnk.get(dn, 999), dn, idx)
                        for idx in range(len(work))
                        if m_min_sp_cache[idx] > U_rank
                        for dn in dom_names_cache[idx]
                    ],
                    reverse=True
                )
                if not eligible:
                    continue

                start   = 0
                matched = False

                while start < len(eligible) and not matched:
                    bundle      = []
                    bundle_size = 0.0
                    next_start  = start + 1

                    for j in range(start, len(eligible)):
                        _, dn, m_idx = eligible[j]
                        dc = dom_dat[dn]
                        ds = dc['Loan size ']

                        if ds > upper:
                            next_start = j + 1
                            break

                        bundle.append((dn, m_idx))
                        bundle_size += ds

                        while bundle_size > upper and len(bundle) > 1:
                            rm, _ = bundle.pop(0)
                            bundle_size -= dom_dat[rm]['Loan size ']
                        if bundle_size > upper:
                            bundle.pop(0)
                            bundle_size = 0.0
                            next_start  = j + 1
                            break

                        if bundle_size >= lower:
                            rem  = float(U_size)
                            tdol = 0.0
                            ordered = sorted(
                                bundle,
                                key=lambda b: member_savings_rate(dom_dat[b[0]], dom_is_float),
                                reverse=True
                            )
                            for bn, _ in ordered:
                                if rem <= 0: break
                                dc2      = dom_dat[bn]
                                notional = min(dc2['Loan size '], rem)
                                rem     -= notional
                                term  = min(U['Loan term'], dc2['Loan term'])
                                _, dollars = pair_savings(U, dc2, dom_is_float, notional, term)
                                tdol += dollars

                            b_names = [b[0] for b in bundle]
                            dom_tot = sum(dom_dat[n]['Loan size '] for n in b_names)
                            members = [dom_dat[n] for n in b_names]
                            drop    = party_rate_drop([U], members) if dom_is_float else party_rate_drop(members, [U])
                            sav     = displayed_savings(drop)

                            if sav >= MIN_RATE_DROP:
                                rm_idxs = {b[1] for b in bundle}
                                work    = [m for k, m in enumerate(work) if k not in rm_idxs]

                                dom_trm = min(dom_dat[n]['Loan term'] for n in b_names)
                                tot_sz  = min(U_size, dom_tot) * 2
                                tot_trm = min(U['Loan term'], dom_trm)

                                if dom_is_float:
                                    new_row = [U_name, ', '.join(b_names),
                                               U_size, dom_tot, tot_sz,
                                               U['Loan term'], dom_trm, tot_trm,
                                               sav, round(tdol, 2)]
                                else:
                                    new_row = [', '.join(b_names), U_name,
                                               dom_tot, U_size, tot_sz,
                                               dom_trm, U['Loan term'], tot_trm,
                                               sav, round(tdol, 2)]

                                work.append(new_row)
                                cache_dirty = True
                                matched  = True
                                improved = True
                                print(f"Rebalanced: shifted connection to cover {U_name}")
                            else:
                                worst = bundle[0][0]
                                for k in range(start, len(eligible)):
                                    if eligible[k][1] == worst:
                                        next_start = k + 1
                                        break
                            break

                    start = next_start

        # ── PART 2: UPGRADE BUNDLE MEMBERS ──
        for bundle_col, anchor_col, brnk, bdat, adat, bundle_is_float in [
            (1, 0, fl_rank, fl_data, fx_data, True),
            (0, 1, fx_rank, fx_data, fl_data, False),
        ]:
            matched_b = {n for m in work for n in match_names(m, bundle_col)}
            pool_b    = floating_pool if bundle_is_float else fixed_pool
            avail     = [c for c in pool_b if c['Company name'] not in matched_b]

            for m_idx, m in enumerate(work):
                amembers = match_names(m, anchor_col)
                if len(amembers) != 1:
                    continue
                anchor_name = amembers[0]

                bmembers = match_names(m, bundle_col)
                if len(bmembers) <= 1:
                    continue

                anchor      = adat[anchor_name]
                anchor_size = anchor['Loan size ']
                lower       = anchor_size * (1 - SIZE_LEEWAY)
                upper       = anchor_size * (1 + SIZE_LEEWAY)

                worst_name = max(bmembers, key=lambda n: brnk.get(n, 999))
                worst_rank = brnk.get(worst_name, 999)

                candidates = sorted(
                    [c for c in avail if brnk.get(c['Company name'], 999) < worst_rank],
                    key=lambda c: brnk.get(c['Company name'], 999)
                )
                if not candidates:
                    continue

                for cand in candidates:
                    cand_name   = cand['Company name']
                    new_members = [n for n in bmembers if n != worst_name] + [cand_name]
                    new_size    = sum(bdat[n]['Loan size '] for n in new_members)
                    if not (lower <= new_size <= upper):
                        continue

                    rem  = float(anchor_size)
                    tdol = 0.0
                    for nm in sorted(new_members,
                                     key=lambda n: member_savings_rate(bdat[n], bundle_is_float),
                                     reverse=True):
                        if rem <= 0:
                            break
                        bc       = bdat[nm]
                        notional = min(bc['Loan size '], rem)
                        rem     -= notional
                        term  = min(anchor['Loan term'], bc['Loan term'])
                        _, dollars = pair_savings(anchor, bc, bundle_is_float, notional, term)
                        tdol += dollars

                    dom_tot     = new_size
                    member_objs = [bdat[n] for n in new_members]
                    drop        = party_rate_drop([anchor], member_objs) if bundle_is_float else party_rate_drop(member_objs, [anchor])
                    sav         = displayed_savings(drop)
                    if sav >= MIN_RATE_DROP:
                        sorted_new = sorted(new_members, key=lambda n: brnk.get(n, 999))
                        dom_trm    = min(bdat[n]['Loan term'] for n in new_members)
                        tot_sz     = min(anchor_size, dom_tot) * 2
                        tot_trm    = min(anchor['Loan term'], dom_trm)
                        if bundle_is_float:
                            work[m_idx] = [anchor_name, ', '.join(sorted_new),
                                           anchor_size, dom_tot, tot_sz,
                                           anchor['Loan term'], dom_trm, tot_trm,
                                           sav, round(tdol, 2)]
                        else:
                            work[m_idx] = [', '.join(sorted_new), anchor_name,
                                           dom_tot, anchor_size, tot_sz,
                                           dom_trm, anchor['Loan term'], tot_trm,
                                           sav, round(tdol, 2)]

                        avail = [c for c in avail if c['Company name'] != cand_name]
                        worst_c = bdat.get(worst_name)
                        if worst_c:
                            avail.append(worst_c)

                        improved = True
                        print(f"Bundle upgraded: {worst_name} → {cand_name} (anchor: {anchor_name})")
                        break

    return work


# MAIN MATCHING LOOP
matched_set = set()
matches     = []
n_rounds    = max(len(fixed_pool), len(floating_pool))

for i in range(n_rounds):
    for anchor_pool, other_pool_full, anchor_is_fixed in [
        (fixed_pool,    floating_pool, True),
        (floating_pool, fixed_pool,    False),
    ]:
        if i >= len(anchor_pool):
            continue

        anchor = anchor_pool[i]
        if anchor['Company name'] in matched_set:
            continue

        other_avail = [c for c in reversed(other_pool_full)
                       if c['Company name'] not in matched_set]
        if not other_avail:
            continue

        anchor_size = anchor['Loan size ']
        start_idx   = 0

        while start_idx < len(other_avail):
            bundle = try_form_bundle(anchor_size, other_avail[start_idx:])
            if bundle is None:
                break

            sav_dollar        = bundle_savings(anchor, bundle, anchor_is_fixed)
            bundle_total_size = sum(b['Loan size '] for b in bundle)
            drop              = party_rate_drop([anchor], bundle) if anchor_is_fixed else party_rate_drop(bundle, [anchor])
            sav               = displayed_savings(drop)

            if sav >= MIN_RATE_DROP:
                total_size  = min(anchor_size, bundle_total_size) * 2
                bundle_term = min(b['Loan term'] for b in bundle)
                total_term  = min(anchor['Loan term'], bundle_term)

                if anchor_is_fixed:
                    row = [
                        anchor['Company name'],
                        ', '.join(b['Company name'] for b in bundle),
                        anchor_size, bundle_total_size, total_size,
                        anchor['Loan term'], bundle_term, total_term,
                        sav, round(sav_dollar, 2)
                    ]
                else:
                    row = [
                        ', '.join(b['Company name'] for b in bundle),
                        anchor['Company name'],
                        bundle_total_size, anchor_size, total_size,
                        bundle_term, anchor['Loan term'], total_term,
                        sav, round(sav_dollar, 2)
                    ]

                matches.append(row)
                matched_set.add(anchor['Company name'])
                for b in bundle:
                    matched_set.add(b['Company name'])
                break

            else:
                lowest = bundle[0]
                pos = next(j for j, c in enumerate(other_avail)
                           if c['Company name'] == lowest['Company name'])
                start_idx = pos + 1

# SORT → REBALANCE → RE-SORT  (m[8] = per-side rate drop throughout)
matches.sort(key=lambda m: m[8], reverse=True)
matches = rebalance_connections(matches)
matches.sort(key=lambda m: m[8], reverse=True)

# WRITE MATCHES TO 'Matches' SHEET
headers = ['C Fixed', 'C Floating', 'C Fixed Loan Size', 'C Floating Loan Size', 'Total Size',
           'C1 Loan Term', 'C2 Loan Term', 'Total Term', 'Savings (rate drop)', 'Savings ($)']
sheet_write(matches_sheet.clear)
if matches:
    sheet_write(matches_sheet.update, [headers] + matches, 'A1')

    # SUMMARY (columns K=label, L=total cut, M=per swap/company)
    total_companies_swapped = len({n for m in matches for col in [0, 1] for n in match_names(m, col)})
    total_savings = sum(m[9] for m in matches)
    n_deals = len(matches)
    total_loan_size = sum(m[2] + m[3] for m in matches)
    avg_loan_size   = total_loan_size / total_companies_swapped
    sheet_write(matches_sheet.update,
                [['Total companies swapped',                   total_companies_swapped, ''],
                 ['Total deals',                               n_deals, ''],
                 ['Average loan size',                         round(avg_loan_size, 2), ''],
                 ['Total savings ($)',                         round(total_savings, 2), ''],
                 [f'Our cut ({OUR_CUT*100:g}%)',               round(total_savings * OUR_CUT, 2),      round(total_savings * OUR_CUT / n_deals, 2)],
                 [f'Bank cut ({BANK_CUT*100:g}%)',             round(total_savings * BANK_CUT, 2),     round(total_savings * BANK_CUT / n_deals, 2)],
                 [f'Broker cut ({BROKER_CUT*100:g}%)',         round(total_savings * BROKER_CUT, 2),   round(total_savings * BROKER_CUT / total_companies_swapped, 2)],
                 [f'Business savings ({BUSINESS_CUT*100:g}%)', round(total_savings * BUSINESS_CUT, 2), round(total_savings * BUSINESS_CUT / total_companies_swapped, 2)],
                 [f'Reserves ({RESERVES_CUT*100:g}%)',         round(total_savings * RESERVES_CUT, 2), round(total_savings * RESERVES_CUT / n_deals, 2)]],
                'K1')
else:
    print("No valid matches found above the savings threshold.")

print(f"Written {len(matches)} matches to 'Matches' sheet.")

In [ ]:
# FAKE DATA GENERATOR
import random
from datetime import date

# PARAMETERS
N_COMPANIES  = 50
FIXED_SPLIT  = 0.5
RATE_MID_MIN = 5.75
RATE_MID_MAX = 11.25
GAP_MIN      = 0.5
GAP_MAX      = 1.5
SIZE_MIN     = 50_000
SIZE_MAX     = 300_000
SIZE_MODE    = 130_000
LOAN_TERM    = 10
NAME_PREFIX  = "Fake"
RANDOM_SEED  = None

# SETUP
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)

company_sheet  = spreadsheet.worksheet("Company info")
existing       = company_sheet.get_all_records()
existing_names = {str(r.get('Company name', '')) for r in existing}

start_idx = 1
while f"{NAME_PREFIX}_{start_idx:03d}" in existing_names:
    start_idx += 1

today = date.today().strftime("%d/%m/%Y")

# GENERATE
new_rows = []
for i in range(N_COMPANIES):
    name = f"{NAME_PREFIX}_{start_idx + i:03d}"
    pref = "Fixed" if random.random() < FIXED_SPLIT else "Floating"

    mid        = random.uniform(RATE_MID_MIN, RATE_MID_MAX)
    gap        = random.uniform(GAP_MIN, GAP_MAX)
    fixed_rate = round(mid + gap / 2, 2)
    float_rate = round(mid - gap / 2, 2)

    loan_size = round(random.triangular(SIZE_MIN, SIZE_MAX, SIZE_MODE), -3)

    # Column order: Company name | Loan size  | Loan term | Bank 1 | Lowest Fixed | Bank 2 | Lowest Floating | Preferred Loan Type | Date Entered
    new_rows.append([name, loan_size, LOAN_TERM, '', fixed_rate, '', float_rate, pref, today])

# WRITE
sheet_write(company_sheet.append_rows, new_rows, value_input_option='USER_ENTERED')

print(f"Added {len(new_rows)} fake companies "
      f"({sum(1 for r in new_rows if r[7]=='Fixed')} Fixed, "
      f"{sum(1 for r in new_rows if r[7]=='Floating')} Floating).")
all_rates = [r for row in new_rows for r in (row[4], row[6])]
gaps = [row[4] - row[6] for row in new_rows]
print(f"Rate range: {min(all_rates):.2f}% – {max(all_rates):.2f}%")
print(f"Gap range:  {min(gaps):.2f}% – {max(gaps):.2f}%  |  avg: {sum(gaps)/len(gaps):.2f}%")

In [ ]:
# RANKING DISPLAY
# FIXED POOL
fixed_ranked = sorted(
    [c for c in data if c['Preferred Loan Type'] == 'Fixed'],
    key=lambda c: (c['Lowest Fixed'] - c['Lowest Floating']) * c['Loan size '] * c['Loan term'],
    reverse=True
)
print("=== FIXED POOL RANKINGS (highest score = best) ===")
print(f"{'Rank':<6} {'Company':<15} {'Score':>15}")
print("-" * 38)
for rank, c in enumerate(fixed_ranked, start=1):
    score = (c['Lowest Fixed'] - c['Lowest Floating']) * c['Loan size '] * c['Loan term']
    print(f"{rank:<6} {str(c['Company name']):<15} {score:>15,.2f}")
print()
# FLOATING POOL
floating_candidates = [c for c in data if c['Preferred Loan Type'] == 'Floating']
min_raw_diff = min(c['Lowest Floating'] - c['Lowest Fixed'] for c in floating_candidates)
floating_ranked = sorted(
    floating_candidates,
    key=lambda c: (c['Lowest Floating'] - c['Lowest Fixed'] - min_raw_diff + 1) * c['Loan size '] * c['Loan term'],
    reverse=True
)
print("=== FLOATING POOL RANKINGS (highest score = best) ===")
print(f"{'Rank':<6} {'Company':<15} {'Score':>15}")
print("-" * 38)
for rank, c in enumerate(floating_ranked, start=1):
    score = (c['Lowest Floating'] - c['Lowest Fixed'] - min_raw_diff + 1) * c['Loan size '] * c['Loan term']
    print(f"{rank:<6} {str(c['Company name']):<15} {score:>15,.2f}")